In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd
import networkx as nx

In [2]:
DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'

# Aims outline

Use same as first attempt but replace linear function with MML functon from LEMBAS

### Load datasets

In [3]:
net = pd.read_csv(f"{DATA_ROOT}/Full data files/network(full).tsv", sep='\t')

In [4]:
network = nx.from_pandas_edgelist(net, 
                             source='TF', 
                             target='Gene', 
                             edge_attr='Interaction')

In [5]:
gene_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

In [6]:
TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)

### Creating dataset object

In [7]:
#torch tutorial code to use accelerator when available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [8]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, TF_expressions, gene_expressions, target_gene, network, transform=None, target_transform=None):
        '''
        Custom dataset for loading expression data across all TFs and samples and one target gene expression across all samples
        For a given index will return all TF expressions in one sample and target gene expression across all samples

        Parameters
        --------------
        device : torch device
            torch device to put dataset on, must be the same device as the model
        TF_expressions : Pandas dataframe
            pandas dataframe of TF expressions where columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        gene_expressions : Pandas dataframe
            pandas dataframe of target gene expressions, columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        target_gene : String
            String containing name of target gene for given model.
        network : networkx network
            Undirected network of the singalling pathway of interest. Used to filter TFs used when predicting target gene expression.
        '''
        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #load network
        self.network = network
        self.TF_expressions = TF_expressions
        self.gene_expressions = gene_expressions

        #filter genes to nodes in network
        self.TF_expressions = self.TF_expressions[[gene for gene in self.TF_expressions.columns if gene in list(self.network.nodes)]]
        self.gene_expressions = self.gene_expressions[[gene for gene in self.gene_expressions.columns if gene in list(self.network.nodes)]] 

        #subset to just target gene of interest
        self.target_gene = target_gene
        self.gene_expressions = self.gene_expressions[self.target_gene]

        #find genes in path and filter TFs accordingly
        self.path_genes = self.genes_in_path(self.network, list(self.TF_expressions.columns), self.target_gene)
        self.TF_expressions = self.TF_expressions [[gene for gene in self.TF_expressions .columns if gene in list(self.path_genes)]]

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def genes_in_path(self, graph, source_nodes, target_node):
        '''
        Paramaters
        -------------
        parameter : graph
            networkx graph
        parameter : soruce_nodes
            list of node labels you wish to use as source nodes
        target_node : string
            label of desired target node
        
        Returns
        -------------
        nodes : List
            List of node labels in the network that are present in at least on shortest path to the specified target node from any of the source nodes.
        '''
        #keep track of all unique nodes found in a shortest path
        node_set = set()
        for source_node in source_nodes:
            #catch errors where there a no path from source to target
            try:
                sps = list(nx.all_shortest_paths(graph, 
                                source=source_node, 
                                target=target_node,
                                weight = None))
            except nx.NetworkXNoPath:
                print(f'No path from {source_node} to {target_node}')

            for path_nodes in sps:
                node_set.update(set(path_nodes))

        return(list(node_set))

    def __len__(self):
        #length of the dataset is the number of samples (not TFs in the dataset) - 15935
        return self.TF_expressions.shape[1]

    def __getitem__(self, idx):
        #get all TFs from sample correspinding to index
        TFs_exp = self.TF_expressions[:, idx]
        #get the target gene expression for the target model
        Gene_exp = self.gene_expressions[idx]
        #returns TFs for sample idx and target gene for sample idx
        return TFs_exp, Gene_exp

In [9]:
#initialise an instance of the dataset object - pass device so tensors and model on same device
dataset = CustomTFGE(device, TF_expressions=TF_expressions, gene_expressions=gene_expressions, network = network, target_gene = 'AADAT')
dataset

No path from DBX1 to AADAT
No path from ZNF512B to AADAT


In [10]:
#new way to create train and test dataset with pytorches dataset objects
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [11]:
#defining the neural network
class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #self.flatten = nn.Flatten()
        self.linear_layer = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the target gene for the same sample
            nn.Linear(1195, 1)
        )
    
    def forward(self, x):
        #forward pass is simply the linear layer
        expressions = self.linear_layer(x)
        return(expressions)

    

In [12]:
#put model on same device as the tensors
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (linear_layer): Sequential(
    (0): Linear(in_features=1195, out_features=1, bias=True)
  )
)


## Train test loop

In [ ]:
def train_loop(dataloader, model, loss_fn, optimiser):
    losses = []
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):     
        #zero the gradient for each batch
        optimiser.zero_grad()

        #create a prediction
        pred = model(X)
        
        #calculate prediction loss
        loss = loss_fn(pred, y)

        #backpropogate for given loss
        loss.backward()
        optimiser.step()

        
        loss = loss.item()
        losses.append(loss)

        if batch +1 == size:
            print(f'Epoch Finished at batch {batch}')

        #print loss every 500 batches - this is effectively every 500th sample
        if batch % 500 == 0:
            print(f"loss: {round(loss, 10)}")
        
    return(losses)



In [33]:
#intialise hyperparameters - batch size is number of samples so that each backprop is done with the entire dataset (gradient descent not stochastic gradient descent)
#dataset is small enough for this to be fine
#investigate hyperparam tuning later
learning_rate = 1e-3
#run 1 sample at a time, but run through each sample per training epoch
batch_size = 1
epochs = 1

#initialize MSE loss function - same as LEMBAS
loss_fn = nn.MSELoss()

#initialise same optimiser as LEMBAS
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [37]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [38]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    losses = train_loop(train_dataloader, model, loss_fn, optimizer)
print("Done!")

Epoch 1
-------------------------------
loss: 0.2034100145
loss: 0.0132369045
loss: 1.0981160402
loss: 0.4032359719
loss: 0.0443681367
loss: 0.2016607374
loss: 0.5463635921
loss: 0.0084736552
loss: 0.0173329711
loss: 0.0147774024
loss: 0.0025241515
loss: 0.2386528701
loss: 0.0157789197
loss: 0.1016994044
loss: 0.7825383544
loss: 0.6496338844
loss: 0.2978497148
loss: 0.0018948546
loss: 0.0090967258
loss: 0.0203748085
loss: 0.3306443989
loss: 0.0010878511
loss: 0.0242855791
loss: 0.0051336242
loss: 0.0023700739
loss: 0.2627378106
Epoch Finished at batch 12747
Done!


# Saving model

In [ ]:
torch.save(model, 'models/linear_with_network.pth')

In [ ]:
#How to load for reference
model = torch.load('models/linear_with_network.pth', weights_only=False)